# Oppgave 3 — Analyse av multi-respons NIR-datasett (Sugar)
**MLA310 — Project 1**

- Treningsdata: 125 sukkerprøver med NIR-spektre (X) og 3 responser (Y)
- Testdata: 21 prøver
- Responser tilsvarer eksperimentelle designinnstillinger for prøveproduksjon

Modelltyper som sammenlignes: **PCR**, **PLS**, **Ridge Regression (RR)**

In [ ]:
using LinearAlgebra, Statistics, MAT, Plots
include("bidiag2.jl")
include("svdr.jl")
include("TregsRLooCV.jl")

## 3a) Innlesning av data

In [ ]:
vars = matread("Sugar.mat")
println("Nøkler i Sugar.mat: ", keys(vars))

In [ ]:
# Tilpass variabelnavn basert på hva som faktisk finnes i filen
# Typiske navn: Xcal/Xtest, Ycal/Ytest eller Xtrain/Xtest
X_train = Float64.(vars["Xtrain"])
Y_train = Float64.(vars["Ytrain"])
X_test  = Float64.(vars["Xtest"])
Y_test  = Float64.(vars["Ytest"])

m_tr, n = size(X_train)
m_te    = size(X_test, 1)
ny      = size(Y_train, 2)
println("Treningsdata: $m_tr prøver, $n bølgelengder, $ny responser")
println("Testdata:     $m_te prøver")

## 3.1) Modelleringstrategi

**Mål:** Velge antall komponenter (PCR/PLS) eller regulariseringsparameter (RR) uten å bruke testdata.

**Strategi:** Leave-one-out kryss-validering (LooCV) på treningsdataene.

- **PCR:** PCA-scores brukes som prediktorer. SVD av X beregnes én gang; LooCV via hatmatrise.
- **PLS:** LooCV med bidiag2-algoritmen (brute-force over treningsprøver).
- **RR:** LooCV via `TregsRLooCV` (analytisk hatmatrise — ingen brute-force nødvendig).

**Multi-respons:** Alle tre responser modelleres simultant. For PCR og RR deler alle responser samme SVD av X.

## 3.2) PCR — Principal Component Regression
### LooCV via SVD og hatmatrise

In [ ]:
# PCR LooCV for én respons
# For PCR med k komponenter: ŷ = T_k * (T_k'T_k)^{-1} T_k' y
# Men T_k kolonner er ortonormale => H_k = U_k * U_k'
# Leverage: h_ii^{(k)} = sum_{j=1}^{k} U[i,j]^2 + 1/m

function pcr_loocv_multi(X, Y, k_max)
    m, n = size(X)
    m, ny = size(Y)
    x̄ = mean(X, dims=1)
    Ȳ = mean(Y, dims=1)
    Xc = X .- x̄
    Yc = Y .- Ȳ

    U, σ, V = svdr(Xc)
    r = length(σ)
    k_max = min(k_max, r)

    rmsecv = zeros(k_max, ny)
    ypred_all = zeros(m, k_max, ny)  # kryss-validerte prediksjoner

    for k in 1:k_max
        Uk = U[:, 1:k]
        # Regresjonskoeffisienter i score-rom
        # PCR: b = V_k Σ_k^{-1} U_k' y  (men med sentrering)
        # Leverage
        h = vec(sum(Uk.^2, dims=2)) .+ 1/m
        for j in 1:ny
            yj = Yc[:, j]
            # Fitted values
            ŷ = Uk * (Uk' * yj)
            resid = yj - ŷ
            # LooCV correction
            ypred_cv = (ŷ .+ Ȳ[j]) - resid ./ (1 .- h)
            ypred_all[:, k, j] = ypred_cv
            rmsecv[k, j] = sqrt(mean((Y[:, j] .- ypred_cv).^2))
        end
    end
    return rmsecv, ypred_all, U, σ, V
end

In [ ]:
k_max_pcr = 30
rmsecv_pcr, ypred_pcr_cv, U_tr, σ_tr, V_tr = pcr_loocv_multi(X_train, Y_train, k_max_pcr)

best_k_pcr = [argmin(rmsecv_pcr[:, j]) for j in 1:ny]
println("PCR — optimale antall komponenter per respons: ", best_k_pcr)
println("PCR — RMSECV:")
for j in 1:ny
    println("  Respons $j: $(round(rmsecv_pcr[best_k_pcr[j], j], digits=4)) (k=$(best_k_pcr[j]))")
end

plot(1:k_max_pcr, rmsecv_pcr,
     xlabel="Antall PC-er", ylabel="RMSECV",
     title="PCR LooCV", label=["Respons 1" "Respons 2" "Respons 3"],
     legend=:topright)

## 3.2b) PLS — Partial Least Squares

In [ ]:
# PLS LooCV for én respons (brute-force)
function pls_loocv_single(X, y, mc_max)
    m = size(X, 1)
    ypred = zeros(m, mc_max)
    for i in 1:m
        idx = [1:i-1; i+1:m]
        β₀_i, β_i = bidiag2(X[idx, :], y[idx]; mc=mc_max)
        for k in 1:mc_max
            ypred[i, k] = (X[i:i, :] * β_i[:, k] .+ β₀_i[:, k])[1]
        end
    end
    rmsecv = sqrt.(mean((y .- ypred).^2, dims=1)[:])
    return rmsecv, ypred
end

In [ ]:
k_max_pls = 20
rmsecv_pls = zeros(k_max_pls, ny)
ypred_pls_cv = zeros(m_tr, k_max_pls, ny)
best_k_pls = zeros(Int, ny)

for j in 1:ny
    println("PLS LooCV for respons $j...")
    @time rmsecv_pls[:, j], ypred_pls_cv[:, :, j] = pls_loocv_single(X_train, Y_train[:, j], k_max_pls)
    best_k_pls[j] = argmin(rmsecv_pls[:, j])
end

println("\nPLS — optimale antall komponenter per respons: ", best_k_pls)
println("PLS — RMSECV:")
for j in 1:ny
    println("  Respons $j: $(round(rmsecv_pls[best_k_pls[j], j], digits=4)) (k=$(best_k_pls[j]))")
end

plot(1:k_max_pls, rmsecv_pls,
     xlabel="Antall PLS-komponenter", ylabel="RMSECV",
     title="PLS LooCV", label=["Respons 1" "Respons 2" "Respons 3"])

## 3.2c) Ridge Regression via TregsRLooCV

In [ ]:
λs = 10 .^ range(-5, 2, length=10)

rmsecv_rr = zeros(length(λs), ny)
λ_opt_rr  = zeros(ny)
bλ_rr     = zeros(n+1, ny)

for j in 1:ny
    press_j, minid_j, _, _, _, _, _, λ_j, bλ_j, _ = TregsRLooCV(X_train, Y_train[:, j], λs)
    rmsecv_rr[:, j] = sqrt.(press_j ./ m_tr)
    λ_opt_rr[j] = λ_j
    bλ_rr[:, j] = bλ_j
    println("Respons $j: λ_opt=$(round(λ_j, sigdigits=3)), RMSECV=$(round(rmsecv_rr[minid_j, j], digits=4))")
end

plot(log10.(λs), rmsecv_rr,
     xlabel="log₁₀(λ)", ylabel="RMSECV",
     title="Ridge Regression LooCV", label=["Respons 1" "Respons 2" "Respons 3"],
     marker=:circle)

## 3.2d) Prediksjon på testdata og RMS-feil

In [ ]:
x̄_tr = mean(X_train, dims=1)
Ȳ_tr = mean(Y_train, dims=1)
Xtr_c = X_train .- x̄_tr
Xte_c = X_test  .- x̄_tr

rms_test_pcr = zeros(ny)
rms_test_pls = zeros(ny)
rms_test_rr  = zeros(ny)

for j in 1:ny
    # --- PCR ---
    k = best_k_pcr[j]
    Uk = U_tr[:, 1:k];  Vk = V_tr[:, 1:k]
    b_pcr = Vk * (Uk' * (Y_train[:, j] .- Ȳ_tr[j]))
    ypred_te_pcr = Xte_c * b_pcr .+ Ȳ_tr[j]
    rms_test_pcr[j] = sqrt(mean((Y_test[:, j] .- ypred_te_pcr).^2))

    # --- PLS ---
    k_p = best_k_pls[j]
    β₀_p, β_p = bidiag2(X_train, Y_train[:, j]; mc=k_p)
    ypred_te_pls = Xte_c * β_p[:, k_p] .+ β₀_p[:, k_p][1]
    rms_test_pls[j] = sqrt(mean((Y_test[:, j] .- ypred_te_pls).^2))

    # --- Ridge ---
    # bλ_rr[:,j] = [intercept; coefficients]
    ypred_te_rr = X_test * bλ_rr[2:end, j] .+ bλ_rr[1, j]
    rms_test_rr[j] = sqrt(mean((Y_test[:, j] .- ypred_te_rr).^2))
end

println("=== RMS prediksjonsfeil på testdata ===")
println("       Resp1    Resp2    Resp3")
println("PCR:  ", round.(rms_test_pcr, digits=4))
println("PLS:  ", round.(rms_test_pls, digits=4))
println("Ridge:", round.(rms_test_rr, digits=4))

# Stolpediagram
bar(["PCR" "PLS" "Ridge"], [rms_test_pcr'; rms_test_pls'; rms_test_rr'],
    xlabel="Metode", ylabel="RMS testfeil",
    title="Sammenligning på testdata",
    label=["Resp 1" "Resp 2" "Resp 3"],
    group=repeat(["PCR","PLS","Ridge"], inner=ny),
    legend=:topright)

## 3.3) Multi-respons TregsRLooCV

### Hvorfor kreves bare én SVD?

Leverage-verdiene $h_{ii}$ i `TregsRLooCV` avhenger **bare av X** (via $U$ og $\sigma$), ikke av $y$:
$$h_{ii}(\lambda) = \sum_j u_{ij}^2 \cdot \frac{\sigma_j}{\sigma_j + \lambda/\sigma_j} + \frac{1}{m}$$

Siden SVD av $X$ er felles for alle responser, og $H$ ikke avhenger av $Y$,  
trenger vi bare **én SVD** av $X$ for å håndtere alle $n_y$ responser simultant.

For regresjonskoeffisientene er det en enkel matrisemultiplikasjon for hvert $\lambda$:
$$B(\lambda) = V \cdot \text{diag}(1/(\sigma + \lambda/\sigma)) \cdot U^T Y$$
som beregner alle responser på én gang.

In [ ]:
# Multi-respons versjon av TregsRLooCV
function TregsRLooCV_multi(X, Y, λs)
    m, n  = size(X)
    m, ny = size(Y)
    λs = reshape(λs, 1, length(λs))
    x̄ = mean(X, dims=1);  Ȳ = mean(Y, dims=1)
    Yc = Y .- Ȳ
    Xc = X .- x̄

    # Én SVD for alle responser
    U, σ, V = svdr(Xc)
    σ_plus_λs_over_σ = σ .+ (λs ./ σ)   # r × nλ

    # Hatmatrise: m × nλ  (felles for alle responser!)
    H = (U .^ 2) * (σ ./ σ_plus_λs_over_σ) .+ 1/m

    # Regresjonskoeffisienter og PRESS for hver respons
    nλ = length(λs)
    press  = zeros(nλ, ny)
    bcoefs = zeros(n, nλ, ny)  # n × nλ × ny
    minids = zeros(Int, ny)
    λ_opts = zeros(ny)
    Bλ     = zeros(n + 1, ny)   # [intercept; slope] for optimale λ

    UᵀY = U' * Yc   # r × ny  (beregnes én gang)

    for j in 1:ny
        # Koeffisienter for alle λ: n × nλ
        bcoefs_j = V * (UᵀY[:, j:j] ./ σ_plus_λs_over_σ)  # n × nλ
        bcoefs[:, :, j] = bcoefs_j

        # Predikerte verdier i score-rom: m × nλ
        Ŷj = U * (σ .* UᵀY[:, j:j] ./ σ_plus_λs_over_σ)  # m × nλ
        resid = Yc[:, j:j] .- Ŷj  # m × nλ

        # PRESS
        press[:, j] = vec(sum((resid ./ (1 .- H)).^2, dims=1))

        minids[j] = argmin(press[:, j])
        λ_opts[j] = λs[minids[j]]
        Bλ[1,   j] = (Ȳ[j] - x̄ * bcoefs_j[:, minids[j]])[1]
        Bλ[2:end, j] = bcoefs_j[:, minids[j]]
    end

    return press, minids, λ_opts, Bλ, H, bcoefs, U, σ, V
end

In [ ]:
press_m, minids_m, λ_opts_m, Bλ_m, H_m, bcoefs_m, U_m, σ_m, V_m =
    TregsRLooCV_multi(X_train, Y_train, λs)

rmsecv_rr_m = sqrt.(press_m ./ m_tr)

println("Multi-respons Ridge LooCV:")
for j in 1:ny
    println("  Respons $j: λ_opt=$(round(λ_opts_m[j], sigdigits=3)), RMSECV=$(round(rmsecv_rr_m[minids_m[j], j], digits=4))")
end

# Verifisering: skal matche univariat TregsRLooCV
println("\nVerifisering (multi vs univariat skal være identisk):")
for j in 1:ny
    diff_press = norm(press_m[:, j] - vec(TregsRLooCV(X_train, Y_train[:, j], vec(λs))[1])) /
                 norm(press_m[:, j])
    println("  Respons $j: relativ PRESS-forskjell = $(round(diff_press, sigdigits=3))")
end

In [ ]:
# Prediksjon på testdata med multi-respons modell
rms_test_rr_m = zeros(ny)
for j in 1:ny
    ypred_te = X_test * Bλ_m[2:end, j] .+ Bλ_m[1, j]
    rms_test_rr_m[j] = sqrt(mean((Y_test[:, j] .- ypred_te).^2))
end

println("=== Endelig sammenligning — RMS på testdata ===")
println("        Resp1    Resp2    Resp3")
println("PCR:   ", round.(rms_test_pcr, digits=4))
println("PLS:   ", round.(rms_test_pls, digits=4))
println("Ridge: ", round.(rms_test_rr, digits=4))
println("Ridge-multi: ", round.(rms_test_rr_m, digits=4))

# Plot
methods = ["PCR", "PLS", "Ridge", "Ridge-multi"]
results = hcat(rms_test_pcr, rms_test_pls, rms_test_rr, rms_test_rr_m)
groupedbar(methods, results',
    ylabel="RMS testfeil",
    title="Sammenligning av metoder på testdata",
    label=["Resp 1" "Resp 2" "Resp 3"],
    legend=:topright)